In [ ]:
import numpy as np
from pathlib import Path
from obspy.core import UTCDateTime, read_inventory
from flovopy.processing.sam import VSAM
import sys
sys.path.append('../week8') # this is where set_samba_data_root.py lives
from set_samba_data_root import DATA_ROOT

DEBUG = True # turns plotting on mostly
window_seconds = 60 # window length for RSAM/DSAM/VSEM in seconds (e.g., 60 for 1-minute metrics)   

# -----------------------------------------------------------------------------
# Load Response information for MVO stations (from SEISAN database)
# -----------------------------------------------------------------------------
RESPONSE_DIR = DATA_ROOT / 'SEISAN_DB' / 'CAL'
from obspy import read_inventory
stationxml = RESPONSE_DIR / 'MV.xml'
inv = read_inventory(stationxml)

# -----------------------------------------------------------------------------
# Initialize Seisan archive client
# -----------------------------------------------------------------------------
from flovopy.research.mvo.archive import MVOSeisanArchive
MVO_ROOT = Path(DATA_ROOT) / "SEISAN_DB"   # root of the MVO Seisan archive
mvo = MVOSeisanArchive(MVO_ROOT)

# Set pre-filter for response removal. Remember this is a bandpass applied directly in the frequency domain
pre_filt = [0.1, 0.2, 18, 25]

# -----------------------------------------------------------------------------
# Define a source location for Soufriere Hills volcano. 
# Station distances to this lat/lon are used to "reduce" the displacement to 1 km distance.
# -----------------------------------------------------------------------------
source = {'lat':16.7164, 'lon':-62.1654}

# --------------------------------------------------------------------------------
# ASL setup
# --------------------------------------------------------------------------------

import asl_env
from flovopy.asl.grid import Grid
from flovopy.asl.wrappers import run_single_event #, find_event_files, run_all_events
from flovopy.asl.config import ASLConfig #,  tweak_config
gridobj = Grid.load(asl_env.GRIDFILE_DEFAULT)
topo_kw = {
    "inv": asl_env.INV,
    "add_labels": True,
    "cmap": "gray",
    "region": asl_env.REGION_DEFAULT,
    "dem_tif": asl_env.DEM_DEFAULT,  # basemap shading from your GeoTIFF - but does not actually seem to use this unless topo_color=True and cmap=None
    "title": "Montserrat DEM + Stations",
    "frame": True,
    "dome_location": asl_env.DOME_LOCATION,
}

baseline_cfg = ASLConfig(
    #inventory=asl_env.INV,
    inventory=inv,
    output_base=asl_env.OUTPUT_DIR,
    gridobj=gridobj,
    global_cache=asl_env.GLOBAL_CACHE,
    station_correction_dataframe=asl_env.station_corrections_df,
    wave_kind="surface",
    speed=1.5,
    Q=23, 
    peakf=2.0,
    dist_mode="3d", 
    misfit_engine="r2",
    window_seconds=window_seconds,
    min_stations=5,
    sam_class=VSAM, 
    sam_metric="mean",
    debug=DEBUG,
)
baseline_cfg.build()

# -----------------------------------------------------------------------------
# Define time range for processing
# -----------------------------------------------------------------------------
startTime = UTCDateTime(2003, 7, 1)   # start date (inclusive)
endTime   = UTCDateTime(2003, 7, 14)  # end date (exclusive)
taperSeconds = 60 * 60
window_seconds = 60  # window length for RSAM/DSAM/VSEM in seconds (e.g., 60 for 1-minute metrics)

# Number of seconds in one day (used for stepping through time)
secondsPerDay = 60 * 60 * 24

# Total number of days (not strictly needed, but useful for reference/debugging)
numDays = (endTime - startTime) / secondsPerDay

# Initialize loop variable
daytime = startTime

# -----------------------------------------------------------------------------
# Loop over each day and compute RSAM
# -----------------------------------------------------------------------------
while daytime < endTime:

    # -------------------------------------------------------------------------
    # Step 1: Load waveform data from Seisan archive for one day
    # -------------------------------------------------------------------------
    print("=" * 80)
    print(f"Reading day: {daytime.date}")

    st = mvo.read_continuous_stream(
        daytime - taperSeconds,
        daytime + secondsPerDay + taperSeconds,
        verbose=False,
        seismic_only=True,
        vertical_only=True,
        merge=True,          # use smart_merge during read
    )

    print(f'- got {len(st)} Trace ids')

    # -------------------------------------------------------------------------
    # Step 2: Pre-process the data (detrend, taper, filter)
    # -------------------------------------------------------------------------    
    for tr in st:
        tr.data = np.nan_to_num(tr.data, nan=0) # Ensure data is in float64 format for remove_response
    st.detrend('linear')  # remove linear trend
    secondsInStream = st[0].stats.endtime - st[0].stats.starttime
    st.merge(method=1, fill_value='latest')  # merge traces, filling gaps with last value
    st.detrend('demean')  # remove mean
    st.taper(max_percentage=taperSeconds/secondsInStream)  # apply a taper to the edges

    # -------------------------------------------------------------------------
    # Step 3: Remove the instrument response to get velocity in meters/second
    # -------------------------------------------------------------------------
    st.remove_response(inventory=inv, pre_filt=pre_filt, output="VEL", taper=False, plot=True, detrend=False) 
    st.trim(starttime=daytime, endtime=daytime + secondsPerDay)
    if DEBUG:
        print(f'Plotting raw velocity seismograms for {daytime}')
        st.plot();
    
    # -------------------------------------------------------------------------
    # Step 4: Run ASL
    # -------------------------------------------------------------------------
    result = run_single_event(
        event_input=st,
        cfg=baseline_cfg,
        refine_sector=False,
        station_gains_df=None,
        switch_event_ctag = True,
        topo_kw=topo_kw,
        mseed_units='m/s', # default units for miniseed files being used - probably "Counts" or "m/s"        
        reduce_time=True,
        debug=DEBUG,
    )

    # -------------------------------------------------------------------------
    # Step forward one day
    # -------------------------------------------------------------------------
    daytime += secondsPerDay